# **Importing Libraries**

In [ ]:
# (Built-in and system libraries)
import os
import random
import zipfile
from io import BytesIO

# Data Science
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Web
import requests

# Image Proccessing
!pip install opencv-python # Install the correct package
import cv2 # Import the library
from PIL import Image

# TensorFlow and Keras
!pip install tensorflow
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # Remove 'image'
from tensorflow.keras.applications import Xception, MobileNetV2, EfficientNetB0, EfficientNetB3
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Scikit-Learn
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score, classification_report

# Google Colab
from google.colab import files

# **Load Data**

In [ ]:
# !pip -q install roboflow

# from roboflow import Roboflow
# rf = Roboflow(api_key="ydflWRS9udM9GpC10fEe")
# project = rf.workspace("aswath-bqgvn").project("multi-retinal-disease-classifica")
# version = project.version(2)
# dataset = version.download("folder")

In [ ]:
!pip -q install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="ydflWRS9udM9GpC10fEe")
project = rf.workspace("student-sqzes").project("eye-disease-fvopu")
version = project.version(5)
dataset = version.download("folder")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.2/85.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 61.5 MB/s eta 0:00:00
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Eye-Disease-5 in folder:: 100%|██████████| 6134/6134 [00:00<00:00, 6470.90it/s]


### Path of dataset

In [ ]:
print(dataset.location)

/content/Eye-Disease-5


In [ ]:
for split in ["train", "valid", "test"]:
    split_total = 0
    # Assuming 'dataset' from your Roboflow download is available:
    data_path = dataset.location # Assign data_path to the location of the downloaded dataset
    folder_path = os.path.join(data_path, split)
    for class_folder in os.listdir(folder_path):
        class_path = os.path.join(folder_path, class_folder)
        if os.path.isdir(class_path):
            count = len(os.listdir(class_path))
            split_total += count
            print(f"{split}/{class_folder}: {count}")
    print(f"{split} total: {split_total}\n")

train/1_normal: 1270
train/2_glaucoma: 1320
train/3_retina_disease: 788
train/2_cataract: 1484
train total: 4862

valid/1_normal: 161
valid/2_glaucoma: 146
valid/3_retina_disease: 90
valid/2_cataract: 141
valid total: 538

test/1_normal: 191
test/2_glaucoma: 202
test/3_retina_disease: 116
test/2_cataract: 208
test total: 717



# **Data Augmentation + Normalization**

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# إعداد مسارات البيانات
train_dir = dataset.location + "/train"
valid_dir = dataset.location + "/valid"
test_dir  = dataset.location + "/test"

# Augmentation + Normalization
train_datagen = ImageDataGenerator(rescale=1./255, horizontal_flip=True, rotation_range=20, zoom_range=0.2)
val_datagen = ImageDataGenerator(rescale=1./255)

# تحميل الصور كمجلدات مصنفة
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    valid_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_generator = val_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)


Found 4862 images belonging to 4 classes.
Found 538 images belonging to 4 classes.
Found 717 images belonging to 4 classes.


#**YOLO Model**

## **YOLOv5**

In [ ]:
!git clone https://github.com/ultralytics/yolov5  # clone
%cd yolov5
%pip install -qr requirements.txt  # install

import torch
import utils
display = utils.notebook_init()  # checks

YOLOv5 🚀 v7.0-416-gfe1d4d99 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)


Setup complete ✅ (2 CPUs, 12.7 GB RAM, 41.0/112.6 GB disk)


### Check on Yolo5

In [ ]:
%cd /content/yolov5/classify
!ls

/content/yolov5/classify
predict.py  train.py  tutorial.ipynb  val.py


In [ ]:
# Ensure we're in the right directory to download our custom dataset
import os
os.makedirs("../datasets/", exist_ok=True)
%cd ../datasets/

/content/yolov5/datasets


In [ ]:
!ls $DATASET_NAME

### Train On Custom Data

In [ ]:
#Save the dataset name to the environment so we can use it in a system call later
dataset_name = dataset.location.split(os.sep)[-1]
os.environ["DATASET_NAME"] = dataset_name

In [ ]:
%cd /content/yolov5/data

/content/yolov5/data


In [ ]:
!apt-get install tree -y
!tree /content/yolov5/datasets/ -L 3
!tree /content/multi-retinal-disease-classifica-2 -L 2

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tree is already the newest version (2.0.2-1).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.
/content/yolov5/datasets/
└── multi-retinal-disease-classifica-2
    ├── README.dataset.txt
    ├── README.roboflow.txt
    ├── test
    │   ├── cataract
    │   ├── diabetic_retinopathy
    │   ├── glaucoma
    │   └── normal
    ├── train
    │   ├── cataract
    │   ├── diabetic_retinopathy
    │   ├── glaucoma
    │   └── normal
    └── valid
        ├── cataract
        ├── diabetic_retinopathy
        ├── glaucoma
        └── normal

16 directories, 2 files
/content/multi-retinal-disease-classifica-2  [error opening dir]

0 directories, 0 files


In [ ]:
# ─── Performance Setup ───
!pip install albumentations
!pip install torchmetrics


In [ ]:
!python /content/yolov5/classify/train.py --model yolov5x-cls.pt --data /content/yolov5/datasets/$DATASET_NAME --epochs 20 --batch-size 16 --img 224 --pretrained weights/yolov5x-cls.pt

2025-04-20 18:47:09.450479: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745174829.490277   18669 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745174829.500828   18669 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
classify/train: model=yolov5x-cls.pt, data=/content/yolov5/datasets/multi-retinal-disease-classifica-2, epochs=20, batch_size=16, imgsz=224, nosave=False, cache=None, device=, workers=8, project=../runs/train-cls, name=exp, exist_ok=False, pretrained=weights/yolov5x-cls.pt, optimizer=Adam, lr0=0.001, decay=5e-05, label_smoothing=0.1, cutoff=None, dropout=None, verbose=False, seed=0, local_rank=-1
github: up to date with https://githu

### Validate Your Custom Model

In [ ]:
# Force same class-to-index mapping as training
#dataset.class_to_idx = train_dataset.class_to_idx

In [ ]:
!python /content/yolov5/classify/val.py --weights /content/yolov5/runs/train-cls/exp/weights/best.pt --data /content/yolov5/datasets/$DATASET_NAME


classify/val: data=/content/yolov5/datasets/multi-retinal-disease-classifica-2, weights=['/content/yolov5/runs/train-cls/exp/weights/best.pt'], batch_size=128, imgsz=224, device=, workers=8, verbose=True, project=../runs/val-cls, name=exp, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 v7.0-416-gfe1d4d99 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)

Fusing layers... 
Model summary: 264 layers, 46796724 parameters, 0 gradients, 128.9 GFLOPs
testing:   0% 0/6 [00:00<?, ?it/s]/content/yolov5/classify/val.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=device.type != "cpu"):
testing: 100% 6/6 [00:02<00:00,  2.27it/s]
                   Class      Images    top1_acc    top5_acc
                     all         688        0.84           1
                cataract           3           0           1
    diabetic_retinopathy         198           1          

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator # Importing ImageDataGenerator
import pandas as pd # Import pandas for DataFrame
import os # Import os for path manipulation

# تحديد الجهاز
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# تحميل الموديل وتحديده للوضع eval
model = torch.hub.load('ultralytics/yolov5', 'custom', path='/content/yolov5/runs/train-cls/exp/weights/best.pt')
model.to(device)
model.eval()

# Preprocessing ثابت
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Assuming 'df_test' contains your test data with 'image_path' and 'label' columns
# If not, define it appropriately before proceeding:
# Constructing df_test from your dataset directory
dataset_path = "/content/multi-retinal-disease-classifica-2/"  # Replace with the actual path to your test dataset
image_paths = []
labels = []

for class_name in os.listdir(dataset_path):
    class_path = os.path.join(dataset_path, class_name)
    for image_name in os.listdir(class_path):
        image_path = os.path.join(class_path, image_name)
        image_paths.append(image_path)
        labels.append(class_name)

df_test = pd.DataFrame({'image_path': image_paths, 'label': labels}) # Creating the DataFrame
# df_test = ...  # Your code to create/load df_test
test_gen = ImageDataGenerator(rescale=1./255).flow_from_dataframe( # Defining and instantiating test_gen
    df_test,
    x_col='image_path',
    y_col='label',
    target_size=(224, 224),
    class_mode='categorical',
    color_mode='rgb',
    shuffle=False,
    batch_size=32
)


# Lists لتخزين النتائج
y_true = []
y_pred = []
losses = []

# Evaluation loop
with torch.no_grad():
    for imgs, labels in tqdm(test_gen, desc="Evaluating"):
        # معالجة كل Batch
        inputs = torch.stack([transform(img) for img in imgs])
        inputs = inputs.to(torch.float32).to(device)
        labels = torch.tensor(labels).to(device)

        outputs = model(inputs)

        # الحسابات
        loss = F.cross_entropy(outputs, labels)
        losses.append(loss.item())

        preds = torch.argmax(outputs, dim=1)

        y_pred.extend(preds.cpu().tolist())
        y_true.extend(labels.cpu().tolist())

# تحويل الأسماء
class_names = list(test_gen.class_indices.keys())

# النتائج
print("\n📊 Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

print(f"✅ Accuracy: {accuracy_score(y_true, y_pred)*100:.2f}%")
print(f"❌ Average Loss: {np.mean(losses):.4f}")

### saving the model in drive

In [ ]:
!cp -r "/content/yolov5/runs/train-cls/exp/weights/best.pt" "/content/drive/MyDrive/eye_disease_classification/exp/yolo5"

## **YOLO8**

### **Fine-tuning2**

In [ ]:
# Install necessary packages
!pip -q install ultralytics

# Import libraries
import os
from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.5/983.5 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 105.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultr

In [ ]:
# Train YOLOv8 classification model
model = YOLO("yolov8s-cls.pt")  # Small version, you can also try yolov8m-cls.pt or yolov8n-cls.pt

model.train(
    data=dataset.location,
    imgsz=224,
    epochs=10,
    batch=32
)

100%|██████████| 12.3M/12.3M [00:00<00:00, 46.7MB/s]


Ultralytics 8.3.115 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=classify, mode=train, model=yolov8s-cls.pt, data=/content/Eye-Disease-5, epochs=10, time=None, patience=100, batch=32, imgsz=224, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, line_widt

100%|██████████| 5.35M/5.35M [00:00<00:00, 204MB/s]


AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 25.7±24.2 MB/s, size: 5.0 KB)


train: Scanning /content/Eye-Disease-5/train... 4862 images, 0 corrupt: 100%|██████████| 4862/4862 [00:01<00:00, 4693.70it/s]

...

                   all      0.961          1



10 epochs completed in 0.078 hours.
Optimizer stripped from runs/classify/train/weights/last.pt, 10.3MB
Optimizer stripped from runs/classify/train/weights/best.pt, 10.3MB

Validating runs/classify/train/weights/best.pt...
Ultralytics 8.3.115 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8s-cls summary (fused): 30 layers, 5,080,324 parameters, 0 gradients, 12.5 GFLOPs
WARNING ⚠️ Dataset 'split=val' not found, using 'split=test' instead.
train: /content/Eye-Disease-5/train... found 4862 images in 4 classes ✅ 
val: /content/Eye-Disease-5/test... found 717 images in 4 classes ✅ 
test: /content/Eye-Disease-5/test... found 717 images in 4 classes ✅ 


               classes   top1_acc   top5_acc: 100%|██████████| 12/12 [00:01<00:00,  9.28it/s]


                   all      0.962          1
Speed: 0.1ms preprocess, 0.5ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to runs/classify/train


ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b7dc7b6dd90>
curves: []
curves_results: []
fitness: 0.9811715483665466
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.9623430967330933, 'metrics/accuracy_top5': 1.0, 'fitness': 0.9811715483665466}
save_dir: PosixPath('runs/classify/train')
speed: {'preprocess': 0.09710545188366013, 'inference': 0.46091514644381676, 'loss': 0.00025074058519919065, 'postprocess': 0.0004187866105966803}
task: 'classify'
top1: 0.9623430967330933
top5: 1.0

In [ ]:
# Evaluate the model on validation set
metrics = model.val(data=dataset.location, imgsz=224)

Ultralytics 8.3.115 🚀 Python-3.11.12 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8s-cls summary (fused): 30 layers, 5,080,324 parameters, 0 gradients, 12.5 GFLOPs
WARNING ⚠️ Dataset 'split=val' not found, using 'split=test' instead.
train: /content/Eye-Disease-5/train... found 4862 images in 4 classes ✅ 
val: /content/Eye-Disease-5/test... found 717 images in 4 classes ✅ 
test: /content/Eye-Disease-5/test... found 717 images in 4 classes ✅ 
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 302.7±124.7 MB/s, size: 4.9 KB)


val: Scanning /content/Eye-Disease-5/test... 717 images, 0 corrupt: 100%|██████████| 717/717 [00:00<?, ?it/s]
               classes   top1_acc   top5_acc: 100%|██████████| 23/23 [00:01<00:00, 14.90it/s]


                   all      0.962          1
Speed: 0.3ms preprocess, 0.9ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to runs/classify/train2


### **Classification Report2**

In [ ]:
from ultralytics import YOLO
from sklearn.metrics import classification_report
import os

# Load model
model = YOLO("runs/classify/train/weights/best.pt")

# Run prediction on test set (images directly, not folders)
test_dir = dataset.location + "/test"
results = []

# Loop through each subdirectory (class) within test_dir
for class_name in os.listdir(test_dir):
    class_path = os.path.join(test_dir, class_name)  # Path to the class subdirectory

    # If it's a directory, process images within it
    if os.path.isdir(class_path):
        results.extend(model.predict(source=class_path, imgsz=224, save=False))

# Labels (folders) as class indices
class_names = model.model.names  # e.g., {0: 'cataract', 1: 'diabetic_retinopathy', ...}

# Ground truth and predicted labels
y_true = []
y_pred = []

# Loop through each prediction
for r in results:
    # Get predicted class (top1)
    pred_class = int(r.probs.top1)
    y_pred.append(pred_class)

    # Get true class from folder name (assuming folder structure: test/classname/image.jpg)
    true_class_name = os.path.basename(os.path.dirname(r.path))
    true_class_index = list(class_names.values()).index(true_class_name)
    y_true.append(true_class_index)

# Print classification report
print(classification_report(y_true, y_pred, target_names=list(class_names.values())))


image 1/191 /content/Eye-Disease-5/test/1_normal/0004_png_jpg.rf.2548996ffce37669886607d115512839.jpg: 224x224 1_normal 0.83, 2_glaucoma 0.17, 2_cataract 0.00, 3_retina_disease 0.00, 4.1ms
image 2/191 /content/Eye-Disease-5/test/1_normal/0005_png_jpg.rf.deabd369091c21be17e4d0d0cca44f63.jpg: 224x224 2_glaucoma 0.87, 1_normal 0.13, 3_retina_disease 0.00, 2_cataract 0.00, 4.0ms
image 3/191 /content/Eye-Disease-5/test/1_normal/0008_png_jpg.rf.cae83c82d2c1e3a8bbef4b46d35d5a5a.jpg: 224x224 1_normal 0.80, 3_retina_disease 0.20, 2_cataract 0.00, 2_glaucoma 0.00, 3.1ms
image 4/191 /content/Eye-Disease-5/test/1_normal/0009_png_jpg.rf.8fce5ea19c0b40f620a62129d6f38367.jpg: 224x224 1_normal 1.00, 3_retina_disease 0.00, 2_cataract 0.00, 2_glaucoma 0.00, 3.0ms
image 5/191 /content/Eye-Disease-5/test/1_normal/0010_png_jpg.rf.5fb3117f0d916a39d5717f12de748d0c.jpg: 224x224 1_normal 0.97, 3_retina_disease 0.03, 2_cataract 0.00, 2_glaucoma 0.00, 3.1ms
image 6/191 /content/Eye-Disease-5/test/1_normal/0010_

#**Gradio**

In [ ]:
pip -q install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 5.6 MB/s eta 0:00:00


In [ ]:
import gradio as gr
# from ultralytics import YOLO
from PIL import Image

# تحميل الموديل المدرب
model = YOLO("runs/classify/train/weights/best.pt")  # غيّر المسار لو غيرت مكان الحفظ

# دالة التصنيف
def classify_image(img):
    results = model.predict(img)
    probs = results[0].probs
    top_class = probs.top1  # index لأعلى فئة
    class_name = model.names[top_class]
    confidence = probs.data[top_class].item()
    return f"Class: {class_name}\nConfidence: {confidence:.2f}"

# واجهة Gradio
gr.Interface(
    fn=classify_image,
    inputs=gr.Image(type="pil", label="ارفع صورة للتصنيف"),
    outputs=gr.Textbox(label="نتيجة التصنيف"),
    title="YOLOv8 Image Classifier",
    description="ارفع صورة ليتم تصنيفها باستخدام YOLOv8s المدرب على بياناتك"
).launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8188964a8e872fdd78.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# **EfficientNetB3 Model**

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0 , EfficientNetB3
from tensorflow.keras.optimizers import Adam

### **ARCH**

In [ ]:
base_model= EfficientNetB3(include_top=False,weights="imagenet",input_shape=(224,224,3))
# base_model.trainable = False


model= models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(256,activation='relu'),
    layers.Dense(128,activation='relu'),
    layers.Dense(4,activation='softmax')
])

model.compile(optimizer=Adam(learning_rate= 0.001), loss='categorical_crossentropy', metrics=['accuracy'])

for layer in base_model.layers[400:]:
 layer.trainable = True
print("Total layers:", len(model.layers))
model.summary()

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Total layers: 5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb3 (Functional)     │ (None, 7, 7, 1536)     │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 75264)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │    19,267,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,084,787 (114.76 MB)

 Trainable params: 29,997,484 (114.43 MB)

 Non-trainable params: 87,303 (341.03 KB)

### **Fine-Tuning**

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Initialize early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',     # Metric to monitor
    patience=0,             # Number of epochs with no improvement after which to stop
    restore_best_weights=True)  # Restore the best model weights after stopping

In [ ]:
from tensorflow.keras.optimizers import AdamW
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
# optimizer = AdamW(learning_rate=0.01, weight_decay=0.001)
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss="categorical_crossentropy", optimizer=optimizer,metrics=["accuracy"])
history = model.fit(train_generator, validation_data=val_generator, epochs=10, callbacks=[early_stopping],shuffle=True)

Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.


Epoch 1/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 226s 897ms/step - accuracy: 0.4740 - loss: 1.1887 - val_accuracy: 0.2621 - val_loss: 2.7491
Epoch 2/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 76s 500ms/step - accuracy: 0.6475 - loss: 0.8262 - val_accuracy: 0.2621 - val_loss: 1.4915
Epoch 3/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 69s 453ms/step - accuracy: 0.7439 - loss: 0.6159 - val_accuracy: 0.2621 - val_loss: 2.9674


### **Classification Report**

In [ ]:
# prompt: Make Classification Report on  test_generator

# Predict on the test set
y_pred_prob = model.predict(test_generator)
y_pred = np.argmax(y_pred_prob, axis=1)

# Get true labels
y_true = test_generator.classes

# Get class names
class_names = list(test_generator.class_indices.keys())

# Generate and print the classification report
print(classification_report(y_true, y_pred, target_names=class_names))


Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.


23/23 ━━━━━━━━━━━━━━━━━━━━ 22s 635ms/step
                  precision    recall  f1-score   support

        1_normal       0.33      0.01      0.01       191
      2_cataract       0.29      1.00      0.45       208
      2_glaucoma       0.00      0.00      0.00       202
3_retina_disease       0.00      0.00      0.00       116

        accuracy                           0.29       717
       macro avg       0.16      0.25      0.11       717
    weighted avg       0.17      0.29      0.13       717



Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


# **MobileNetV2 Model**

### **ARCH**

In [ ]:
base_model = tf.keras.applications.MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
avg = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
output = tf.keras.layers.Dense(4, activation="softmax")(avg)
model = tf.keras.Model(inputs=base_model.input, outputs=output)
# base_model.trainable = False
for layer in base_model.layers[120:]:
    layer.trainable = True

print("Total layers:", len(model.layers))
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Total layers: 156


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,263,108 (8.63 MB)

 Trainable params: 2,228,996 (8.50 MB)

 Non-trainable params: 34,112 (133.25 KB)

### **Fine-Tuning**

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Initialize early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',     # Metric to monitor
    patience=0,             # Number of epochs with no improvement after which to stop
    restore_best_weights=True)  # Restore the best model weights after stopping

In [ ]:
from tensorflow.keras.optimizers import AdamW
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
# optimizer = AdamW(learning_rate=0.01, weight_decay=0.001)
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss="categorical_crossentropy", optimizer=optimizer,metrics=["accuracy"])
history = model.fit(train_generator, validation_data=val_generator, epochs=10, callbacks=[early_stopping],shuffle=True)

Epoch 1/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 113s 538ms/step - accuracy: 0.5390 - loss: 1.0614 - val_accuracy: 0.2621 - val_loss: 5.9534
Epoch 2/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 101s 380ms/step - accuracy: 0.7545 - loss: 0.6084 - val_accuracy: 0.2621 - val_loss: 6.5774


### **Classification Report**

In [ ]:
# prompt: Make Classification Report on  test_generator

# Predict on the test set
y_pred_prob = model.predict(test_generator)
y_pred = np.argmax(y_pred_prob, axis=1)

# Get true labels
y_true = test_generator.classes

# Get class names
class_names = list(test_generator.class_indices.keys())

# Generate and print the classification report
print(classification_report(y_true, y_pred, target_names=class_names))


23/23 ━━━━━━━━━━━━━━━━━━━━ 8s 232ms/step
                  precision    recall  f1-score   support

        1_normal       0.00      0.00      0.00       191
      2_cataract       0.29      1.00      0.45       208
      2_glaucoma       0.00      0.00      0.00       202
3_retina_disease       0.00      0.00      0.00       116

        accuracy                           0.29       717
       macro avg       0.07      0.25      0.11       717
    weighted avg       0.08      0.29      0.13       717



Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


In [ ]:
# import tensorflow_hub as hub
# # Load the Vision Transformer model (ViT) from TensorFlow Hub
# model_url = "https://tfhub.dev/google/vit_b16/1"
# vit_layer = hub.KerasLayer(model_url, input_shape=(224, 224, 3))

# # Define the model using the functional API
# inputs = tf.keras.Input(shape=(224, 224, 3))
# x = vit_layer(inputs)
# x = tf.keras.layers.GlobalAveragePooling2D()(x)
# x = tf.keras.layers.Dense(1024, activation='relu')(x)
# x = tf.keras.layers.Dropout(0.5)(x)
# outputs = tf.keras.layers.Dense(4, activation='softmax')(x)


# model = tf.keras.Model(inputs=inputs, outputs=outputs)

# **Xception Model**

### **ARCH**

In [ ]:
base_model = tf.keras.applications.xception.Xception(weights="imagenet",include_top=False, input_shape=(224, 224, 3))
avg = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
output = tf.keras.layers.Dense(4, activation="softmax")(avg)
model = tf.keras.Model(inputs=base_model.input, outputs=output)

for layer in base_model.layers[100:]:
    layer.trainable = True
print("Total layers:", len(model.layers))
model.summary()

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Total layers: 134


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 111, 111,  │        864 │ input_layer_3[0]… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_bn     │ (None, 111, 111,  │        128 │ block1_conv1[0][… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1_act    │ (None, 111, 111,  │          0 │ block1_conv1_bn[… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 109, 109,  │     18,432 │ block1_conv1_act… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_bn     │ (None, 109, 109,  │        256 │ block1_conv2[0][… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2_act    │ (None, 109, 109,  │          0 │ block1_conv2_bn[… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1     │ (None, 109, 109,  │      8,768 │ block1_conv2_act… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv1_bn  │ (None, 109, 109,  │        512 │ block2_sepconv1[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_act │ (None, 109, 109,  │          0 │ block2_sepconv1_… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2     │ (None, 109, 109,  │     17,536 │ block2_sepconv2_… │
│ (SeparableConv2D)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_sepconv2_bn  │ (None, 109, 109,  │        512 │ block2_sepconv2[… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 55, 55,    │      8,192 │ block1_conv2_act… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 55, 55,    │          0 │ block2_sepconv2_… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 55, 55,    │        512 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 55, 55,    │          0 │ block2_pool[0][0… │
│                     │ 128)              │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_sepconv1_act │ (None, 55, 55,    │          0 │ add[0][0]       

 Total params: 20,869,676 (79.61 MB)

 Trainable params: 20,815,148 (79.40 MB)

 Non-trainable params: 54,528 (213.00 KB)

### **Fine-Tuning**

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Initialize early stopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',     # Metric to monitor
    patience=0,             # Number of epochs with no improvement after which to stop
    restore_best_weights=True)  # Restore the best model weights after stopping

In [ ]:
from tensorflow.keras.optimizers import AdamW
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
# optimizer = AdamW(learning_rate=0.01, weight_decay=0.001)
optimizer = tf.keras.optimizers.SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss="categorical_crossentropy", optimizer=optimizer,metrics=["accuracy"])
history = model.fit(train_generator, validation_data=val_generator, epochs=10, callbacks=[early_stopping],shuffle=True)

Epoch 1/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 156s 765ms/step - accuracy: 0.3891 - loss: 1.2822 - val_accuracy: 0.5446 - val_loss: 1.0857
Epoch 2/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 81s 532ms/step - accuracy: 0.6123 - loss: 0.8959 - val_accuracy: 0.5799 - val_loss: 0.9762
Epoch 3/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 82s 540ms/step - accuracy: 0.7194 - loss: 0.7218 - val_accuracy: 0.6468 - val_loss: 0.8782
Epoch 4/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 81s 533ms/step - accuracy: 0.7632 - loss: 0.6038 - val_accuracy: 0.7454 - val_loss: 0.6229
Epoch 5/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 82s 535ms/step - accuracy: 0.8133 - loss: 0.4959 - val_accuracy: 0.7862 - val_loss: 0.5271
Epoch 6/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 82s 539ms/step - accuracy: 0.8406 - loss: 0.4174 - val_accuracy: 0.8197 - val_loss: 0.4325
Epoch 7/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 82s 538ms/step - accuracy: 0.8749 - loss: 0.3390 - val_accuracy: 0.8755 - val_loss: 0.3446
Epoch 8/10
152/152 ━━━━━━━━━━━━━━━━━━━━ 82s 535ms/step - accuracy: 0.9039 - loss: 

### **Classification Report**

In [ ]:
# prompt: Make Classification Report on  test_generator

# Predict on the test set
y_pred_prob = model.predict(test_generator)
y_pred = np.argmax(y_pred_prob, axis=1)

# Get true labels
y_true = test_generator.classes

# Get class names
class_names = list(test_generator.class_indices.keys())

# Generate and print the classification report
print(classification_report(y_true, y_pred, target_names=class_names))


23/23 ━━━━━━━━━━━━━━━━━━━━ 11s 349ms/step
                  precision    recall  f1-score   support

        1_normal       0.21      0.20      0.20       191
      2_cataract       0.31      0.33      0.32       208
      2_glaucoma       0.31      0.30      0.31       202
3_retina_disease       0.16      0.17      0.17       116

        accuracy                           0.26       717
       macro avg       0.25      0.25      0.25       717
    weighted avg       0.26      0.26      0.26       717



# **THANKS**